# 用 YOLOv8 实现迁移学习 

## 什么是迁移学习?

**迁移学习** 是指把在大型数据集(如 COCO,约 12 万张图、80 类)上**预训练好**的模型权重,复用到**自己的数据集**上继续训练。

YOLOv8 预训练时,网络的**浅层**已经学会了通用特征(边缘、纹理、形状),**深层**学会了"目标是什么"。

迁移学习就是"站在巨人的肩膀上"——不用从零学起,只需在已有知识上微调。

## YOLOv8 中迁移学习的两种方式

### 方式一:直接微调 (Fine-tuning)—— 最常用
直接加载预训练权重 `yolov8n.pt`,在自己的数据集上继续训练。
- 训练快、收敛快、**数据量少也能训**
- 适用:任务与 COCO 相近(通用目标检测)

### 方式二:冻结层训练 (Freeze layers)
**冻结骨干网络(浅层通用特征)不更新**,只训练深层/检测头。
- 防止小数据集上"灾难性遗忘",更省显存、更稳
- 适用:数据量很少,或想保留 COCO 学到的通用特征

## 常用迁移学习参数

| 参数 | 作用 | 推荐值 |
|------|------|--------|
| `pretrained=True` | 使用预训练权重 | `True` |
| `freeze=10` | 冻结前 N 层(通常为 backbone) | `10` 或层索引列表 |
| `lr0` | 初始学习率(迁移学习用**较小值**) | `0.001 ~ 0.01` |
| `epochs` | 训练轮数 | `50 ~ 100`(小数据) |
| `data` | 数据集 yaml 路径 | 自定义 |

## 关键技巧:自定义数据集的类别数 ≠ 80 时

COCO 预训练模型是 **80 类**。如果你的数据集类别数不同,最后一层检测头(Detect head)的**结构会改变**,直接 `YOLO("yolov8n.pt")` 加载会权重错位/报错。

**正确做法(两段式):**
1. 先用 `yolov8n.yaml` **重建网络** → 检测头按你的类别数重新初始化
2. 再 `.load("yolov8n.pt")` 把**预训练权重**加载进去(backbone/neck 直接复用,检测头用新初始化)

> 也就是说:`YOLO("yolov8n.yaml").load("yolov8n.pt")` 才是"自定义数据集 + 迁移学习"的标准打开方式,而不是直接 `YOLO("yolov8n.pt")`。

## 常用迁移学习技巧总结

1. **小学习率**:`lr0=0.01` 以下,别让预训练权重被冲掉
2. **冻结浅层**:数据少时 `freeze=10`,先训检测头再解冻
3. **先冻结后解冻**:先用大 `freeze` 跑几轮,再 `freeze=0` 全量微调
4. **恢复训练**:中断后 `resume=True` 从 `last.pt` 接着训


In [ ]:
# ============================================================
# YOLOv8 迁移学习完整示例
# 前提:已安装 ultralytics,或用本地源码:
#   pip install -e ultralytics-8.4.113
# ============================================================
from ultralytics import YOLO

# ---------- 方式一:直接微调 (类别数与 COCO 相同或相近) ----------
# 自动下载 yolov8n.pt 预训练权重 (n=最小最快, 还有 s/m/l/x)
model = YOLO("yolov8n.pt")

model.train(
    data="coco8.yaml",      # ⚠️ 换成你自己的数据集 yaml (含 train/val 路径和 names)
    epochs=50,              # 迁移学习不需要太多轮次就能收敛
    imgsz=640,
    lr0=0.01,               # 迁移学习建议用较小学习率
    pretrained=True,        # 使用预训练权重 (默认 True)
    # freeze=10,            # 方式二:冻结前 10 层 (backbone) 后再训练
)

# ---------- 方式二:冻结层训练 (数据少 / 想保留通用特征) ----------
# model = YOLO("yolov8n.pt")
# model.train(data="coco8.yaml", epochs=50, freeze=10)   # 冻结前10层
# # 也可以传层索引列表,例如冻结 backbone 的指定层:
# # model.train(data="coco8.yaml", epochs=50, freeze=[0, 1, 2, 3, 4, 5])

# ---------- 自定义数据集 (类别数 ≠ 80) 的正确姿势 ----------
# 1) 用 yaml 重建网络 → 检测头按新类别数初始化
# 2) .load() 把预训练权重塞进去 → backbone/neck 直接复用
# model = YOLO("yolov8n.yaml").load("yolov8n.pt")
# model.train(
#     data="my_dataset.yaml",   # 自定义数据集: 比如 3 类水果
#     epochs=100,
#     imgsz=640,
#     lr0=0.005,
#     freeze=10,                # 数据少时先冻结 backbone 训检测头
# )

# ---------- 恢复中断的训练 ----------
# model = YOLO("runs/detect/train/weights/last.pt")
# model.train(resume=True)   # 自动接着上次的配置继续训练

# ---------- 训练完后用微调模型做推理 ----------
# best = YOLO("runs/detect/train/weights/best.pt")
# results = best("test.jpg")   # 或对视频/文件夹推理


# 自定义模块接入 YOLOv8 的标准四步流程

以本次新增 **VGG16 backbone** 为例,总结"自定义模块接入 ultralytics"的完整套路:

##  四步流程总览

```mermaid
graph LR
    A["1. block.py 加类"] --> B["2. __init__.py 导出"]
    B --> C["3. tasks.py 注册"]
    C --> D["4. yaml 引用"]
    D --> E["5. 验证: YOLO(yaml) 加载 + 前向"]
```

##  涉及的 4 个文件

| 步骤 | 文件 | 做什么 |
|------|------|--------|
| ① 加类 | `ultralytics/nn/modules/block.py` | 定义 `class VGG16(nn.Module)`,并加入模块底部 `__all__` |
| ② 导出 | `ultralytics/nn/modules/__init__.py` | 在 block 导入列表中加入 `VGG16` |
| ③ 注册 | `ultralytics/nn/tasks.py` | 顶部 import 引入 + `parse_model` 的 **`base_modules`** 集合加入 `VGG16` |
| ④ 引用 | `ultralytics/cfg/models/v8/yolov8-vgg16.yaml` | backbone 里写 `- [-1, 1, VGG16, [64, 2]]` |

##  关键点:为什么必须在 `base_modules` 注册?

`parse_model` 对 `base_modules` 里的模块会自动处理:
```python
if m in base_modules:
    c1, c2 = ch[f], args[0]      # 自动取上一层的输出通道作为 c1
    c2 = make_divisible(c2 * width, 8)  # 自动应用 scales 宽度缩放
    args = [c1, c2, *args[1:]]   # 自动补齐输入通道参数
```
不注册的话,通道数不会自动推导,模型会构建失败。

##  本次 VGG16 类的设计(仿 ResNetLayer)

```python
class VGG16(nn.Module):
    def __init__(self, c1, c2, n=1, maxpool=True):
        # n 个 3x3 Conv (Conv2d+BN+激活)
        convs = [Conv(c1, c2, k=3, s=1, act=True)]
        convs += [Conv(c2, c2, k=3, s=1, act=True) for _ in range(n - 1)]
        self.convs = nn.Sequential(*convs)
        self.maxpool = nn.MaxPool2d(2, 2) if maxpool else nn.Identity()
    def forward(self, x):
        return self.maxpool(self.convs(x))
```

##  yaml 里的调用方式

```yaml
backbone:
  - [-1, 1, VGG16, [64, 2]]   # 输出64通道, 2个3x3卷积 → P1/2
  - [-1, 1, VGG16, [128, 2]]  # → P2/4
  - [-1, 1, VGG16, [256, 3]]  # → P3/8  ← P3特征给head
  - [-1, 1, VGG16, [512, 3]]  # → P4/16 ← P4特征
  - [-1, 1, VGG16, [512, 3]]  # → P5/32 ← P5特征
  - [-1, 1, SPPF, [512, 5]]   # 感受野增强
```

##  验证方法

```python
from ultralytics import YOLO
m = YOLO("ultralytics-8.4.113/ultralytics/cfg/models/v8/yolov8-vgg16.yaml")
m.info()                       # 143层, 39.2M参数
# 检查层类型: 0-4层应为 VGG16
[print(i, type(l).__name__) for i, l in enumerate(m.model.model)]
# 端到端推理
m("bus.jpg")
```

##  总结

1. **YOLOv8 的架构 = yaml 数据**,`block.py` 里的类都是"积木",自定义模块就是造新积木
2. **注册链路是硬性要求**:加类 → 导出 → 注册,少一步都会报错
3. **类设计建议仿照现有模块**(如 `ResNetLayer`):构造函数接收 `(c1, c2, ...)`,内部用 `nn.Sequential` 组合
4. 想让自定义 backbone **加载官方预训练权重**(如 torchvision vgg16),需要额外写 **state_dict 键名映射**代码(权重键名不同)

# YOLOv8 损失函数详解

> 源码位置:`ultralytics/utils/loss.py` 中的 `v8DetectionLoss`

## 总公式

YOLOv8 的检测损失由 **三部分** 加权求和(权重在超参 `hyp` 里,默认 `box=7.5, cls=0.5, dfl=1.5`):

$$
L_{total} = \underbrace{w_{box} \cdot L_{box}}_{CIoU} \;+\; \underbrace{w_{cls} \cdot L_{cls}}_{BCE} \;+\; \underbrace{w_{dfl} \cdot L_{dfl}}_{DFL}
$$

```python
loss[0] *= self.hyp.box   # box gain = 7.5
loss[1] *= self.hyp.cls   # cls gain = 0.5
loss[2] *= self.hyp.dfl   # dfl gain = 1.5
```

## 1. 分类损失 — BCE(二元交叉熵)

- 用 `nn.BCEWithLogitsLoss`,对每个 anchor 的每个类别做**二分类**(有/无该目标)
- YOLOv8 是 **anchor-free + 多标签分类**:`target_scores` 是 one-hot 软标签(带 IoU 权重)
- 与正样本数 `target_scores_sum` 归一化,避免图/目标数量影响量级

$$
L_{cls} = -\frac{1}{S}\sum_{i}\sum_{c} \big[\, t_{ic}\log\sigma(p_{ic}) + (1-t_{ic})\log(1-\sigma(p_{ic})) \,\big]
$$

## 2. 定位损失 — CIoU(Complete IoU)

- 只对**前景**(`fg_mask`,即被分配为正样本的 anchor)计算
- `bbox_iou(..., CIoU=True)`,在 IoU 基础上加入**中心点距离 + 宽高比**惩罚:

$$
L_{CIoU} = 1 - IoU + \frac{\rho^2(\mathbf{b},\mathbf{b}^{gt})}{c^2} + \alpha v
$$

其中 $\rho^2/c^2$ 是中心点归一化距离,$v$ 衡量宽高比一致性,$\alpha$ 是平衡权重。

> IoU 只关心重叠面积,CIoU 额外要求"框得又准又正",收敛更快更稳。

## 3. DFL — Distribution Focal Loss(分布焦点损失)

- 定位回归不再是"直接回归一个值",而是把每条边(左/右/上/下)**离散成 16 个 bin**(`reg_max=16`),预测一个**概率分布**
- DFL 让分布**集中到目标值附近**的两个 bin 上(左 `tl` + 右 `tr` 加权):

$$
L_{DFL} = -\big[\, w_l\cdot\log(P_{tl}) + w_r\cdot\log(P_{tr}) \,\big]
$$

- 最终坐标 = 分布与 bin 索引的加权期望(`softmax ∘ proj`),能表达"边界其实是个模糊区间",对遮挡/标注噪声更鲁棒

```python
# 解码:分布 → 期望坐标 (loss.py 的 bbox_decode)
pred_dist = pred_dist.view(b, a, 4, c // 4).softmax(3).matmul(self.proj)
```

## 4. 正样本分配 — TaskAlignedAssigner

- 损失只在**被分配为正样本**的 anchor 上算,分配用 `TaskAlignedAssigner`(topk=10)
- 依据:预测与 GT 的 **对齐度** $= \text{score}^{\alpha} \times \text{IoU}^{\beta}$($\alpha=0.5, \beta=6.0$)
- 让"分类好 + 定位准"的 anchor 当正样本,替代旧版基于 IoU 的静态分配

## 小结

| 损失 | 作用 | 实现 |
|------|------|------|
| `cls` (BCE) | 判断每个位置有没有目标、是哪一类 | `nn.BCEWithLogitsLoss` |
| `box` (CIoU) | 把预测框拉向真实框(重叠+中心+宽高比) | `BboxLoss` + `bbox_iou(CIoU=True)` |
| `dfl` (DFL) | 用分布表示边界,回归更精细 | `DFLoss(reg_max=16)` |

---

# 与 YOLOv5 损失的对比

## 总览对比表

| 维度 | YOLOv5 | YOLOv8 |
|------|--------|--------|
| 损失组成 | `box + obj + cls`(三项) | `box + cls + dfl`(三项) |
| 定位损失 | CIoU | CIoU + **DFL**(分布回归) |
| 分类损失 | BCE(硬标签) | BCE(**软标签**,带 IoU 权重) |
| 置信度分支 | 有独立 `obj`(objectness) | **无**——置信度直接取分类分数 |
| 锚框 | **anchor-based**(K-means 预置 3×3 锚) | **anchor-free**(预测中心点到边的距离) |
| 正样本分配 | 静态宽高比匹配(`anchor_t=4.0`) | 动态 `TaskAlignedAssigner`(对齐度 topk) |
| 默认权重 | `box=0.05, obj=1.0, cls=0.5` | `box=7.5, cls=0.5, dfl=1.5` |

## 核心差异

### ① YOLOv5 多一个 obj 置信度分支

- **v5**:每个 anchor 额外预测"这里有没有目标"的置信度,推理时 `conf = obj × cls`(两者都低才判负)
- **v8**:删掉 obj 分支,分类分数本身就承担了置信度职责 → 少一个分支、少一组标签,结构更简洁

### ② YOLOv5 用预置锚框,v8 完全去锚

- **v5**:训练前用 K-means 在数据集上聚类出 3×3 组锚框,GT 按宽高比匹配到锚(比值 < `anchor_t=4.0`)
- **v8**:每个特征图位置直接回归"到上/下/左/右四条边的距离",无需聚类、无需调锚框超参

### ③ DFL 是 YOLOv8 新增的

- **v5**:直接回归 4 个坐标值(单值回归)
- **v8**:每条边回归成 16 bin 的概率分布,再取期望 → 边界更平滑、对小目标更友好

### ④ 标签分配从静态 → 动态

- **v5** `build_targets`:按"GT 与锚宽高比 < 4"一次性静态匹配,训练中不更新
- **v8** `TaskAlignedAssigner`:每个 batch 根据当前预测的"分类分 × IoU"动态选 topk=10,分配随训练自动优化

## 演进路线

```
YOLOv5:  CIoU + obj(BCE) + cls(BCE)           ← anchor-based,有置信度分支
   │  去锚框 / 去 obj / 软标签 / 分布回归 / 动态分配
   ↓
YOLOv8:  CIoU + DFL + cls(BCE 软标签)         ← anchor-free,置信度并入分类
```

> 一句话总结:YOLOv8 保留了 v5 的 **CIoU** 和 **BCE** 两大核心,砍掉了 obj 分支和预置锚框,新增 **DFL** 分布回归,并把正样本分配升级为**动态对齐**——损失项数量没变,但每一项都更"精细化"了。


# block.py 模块速查表(按版本/用途归类)

> `ultralytics/nn/modules/block.py` 当前 **2109 行、55 个类**,是"所有 YOLO 版本积木的一锅烩"。
> 本表帮助快速定位:看到陌生类时,先判断它属于哪个版本/用途。

##  模块分类表

| 版本/用途 | 核心模块 | 一句话说明 |
|-----------|---------|-----------|
| **YOLOv8** 原始核心 | `C2f`, `SPPF`, `Bottleneck`, `C3`, `C3x`, `C3TR`, `DFL`, `Proto`, `SPP`, `C1`, `C2`, `C3Ghost`, `GhostBottleneck` | 最早的一批积木(v8 时代 ~1000 行) |
| **YOLOv9** | `RepNCSPELAN4`, `ADown`, `AConv`, `SPPELAN`, `ELAN1`, `RepCSP`, `RepBottleneck` | GELAN 结构、可重参数化 |
| **YOLO11** | `C3k2`, `C2PSA`, `PSABlock`, `C3k`, `SCDown`, `C2fPSA` | PSA 注意力、轻量下采样 |
| **YOLO12** | `A2C2f`, `AAttn`, `ABlock`, `CIB`, `C2fCIB` | 注意力 + 残差融合 |
| **YOLO26**(最新) | `C3f`, `C3k`, `Proto26`, `SAVPE`, `RealNVP`, `SwiGLUFFN` | 最新一代模块 |
| **RT-DETR / 注意力** | `ImagePoolingAttn`, `MaxSigmoidAttnBlock`, `Attention`, `C2fAttn` | Transformer 注意力系 |
| **YOLO-World / 多任务** | `ContrastiveHead`, `BNContrastiveHead`, `Proto` | 文本-视觉对齐、分割 |
| **ResNet backbone** | `ResNetBlock`, `ResNetLayer` | RT-DETR 的 ResNet 骨干 |
| **其他功能** | `RepVGGDW`(RepVGG), `CBLinear`/`CBFuse`(多backbone), `HGStem`/`HGBlock`(HorNet), `TorchVision` | 杂项扩展 |
| **自定义 🆕** | `VGG16` | 我们刚加的 VGG16 块 |



##  使用建议

- **核心必懂**:`Conv`, `C2f`, `Bottleneck`, `DWConv`, `SPPF` — 这些是 90% 场景用到的积木
- **其余按需查表**:看到陌生类,对照本表判断它属于哪个版本,再决定要不要深入
- **自定义模块**:永远走四步流程(加类 → 导出 → 注册 base_modules → yaml 引用)


# 项目实战:6 类蔬菜图像分类训练总结

> 对应文档:`蔬菜分类训练.md` | 日期:2026-08-03 ~ 2026-08-04
> 框架:YOLOv5-cls(`yolov5s-cls.pt` 预训练) | 环境:Python 3.11 + PyTorch 2.5.1 + RTX 3060 (6GB)

## 任务与数据

- **6 类蔬菜**:`beans`(豆角)、`eggplant`(茄子)、`ladies_finger`(秋葵)、`onion`(洋葱)、`pointed_gourd`(蛇瓜)、`potato`(土豆)
- **白底增强数据**:6 类 × 约 3333 张,按 **8:2** 分层划分 train/val(种子 42)
- **Kaggle 真实照片**:15 类中仅 3 类对应(Bean/Brinjal/Potato),合并进微调集

| 数据集 | train | val | 说明 |
|------|-------|-----|------|
| 白底(veg3) | 19998 | 4998 | 每类 3333/833 |
| 微调合并 | 22998 | 5598 | 3 类加 1000 张真实照 |

## 训练过程(4 次实验)

| 实验 | epochs | batch | 结果 |
|------|--------|-------|------|
| `veg` | 30 | 64 | 中断 |
| `veg2` | 30 | 64 | 中断于 epoch 14 |
| `veg3` | 30 | 64 | ✅ top1 = **100%** |
| `veg4` | 100 | 32 | top1=99.6%,**后期过拟合**(第 8 轮即收敛) |

> 结论:**白底数据简单,10~30 轮足够**,多训反而过拟合。

## 关键问题:域偏移

- 白底模型 top1 100%,但在**真实照片上仅 25%(3/12)**
- 根因:训练数据单一(白底、单物体、固定光照)

## 解决方案:混合真实数据微调 ✅

- 用 `veg3/best.pt` + Kaggle 真实照片,`lr0=0.0001` 在**服务器**训 100 轮
- 结果:top1 = **99.94%**(best 在第 5 轮)
- 同一批 18 张真实照片重测:对应类 **3/12 → 12/12(100%)**,置信度 0.75~0.96

## 部署与工具链

- **ONNX 导出**:`best.onnx`(15.9MB,FP32,224×224,CPU 上 12/12 正确)
- **跨平台修复**:Linux 训练的 `.pt` 含 `PosixPath`,需 `fix_linux_pt.py` 修复后 Windows 才能加载
- **当前最佳模型**:`best_windows.pt`(真实场景 100%,白底 100%)

## 核心经验

1. **迁移学习小学习率**(`lr0=0.0001`)微调,不破坏已学特征
2. **域偏移是分类模型最常见陷阱**——测试集必须贴近真实部署场景
3. **混合真实数据微调**是解决域偏移的有效手段(25% → 100%)
4. 未见过的新类别(卷心菜/番茄等)预测错误属正常——模型只有 6 类